# Bronze | Streaming simulado (micro-lotes com Structured Streaming)

## O que este notebook faz (e o que NÃO faz)

| Componente | Implementação | Observação honesta |
|---|---|---|
| **Mensageria** | Tabela Delta `workspace.bronze.eventos_fila` faz o papel do broker (log append-only) | Não é Kafka/Kinesis. É uma fila simulada, mas produtor e consumidor são **desacoplados** |
| **Produtor** | Função Python grava eventos na fila; `event_time` = instante real de produção | Valores de taxa/meta são **sintéticos** e aleatórios |
| **Consumidor** | `readStream` + `foreachBatch` com `trigger(availableNow=True)` e checkpoint | Processamento incremental em micro-lotes (serverless não oferece trigger contínuo) |
| **Latência** | `processing_time` é carimbado **pelo consumidor na chegada**; latência = `processing_time − event_time` | Há uma espera simulada de transporte (`time.sleep`), mas o tempo medido é o relógio real, não um número sorteado |
| **Duplicados** | Fila reentrega eventos já enviados (inclusive de ciclos anteriores) → *at-least-once* | Dedup por `event_id` **entre lotes** via `MERGE` na Silver |
| **Inválidos** | Taxa nula/fora de [0,100], UF ou rede desconhecidas → ficam na Bronze com `valido=false` e motivo; não vão para Silver/Gold | |
| **Origem** | `origem = 'SIMULADO'` em todas as tabelas; Gold de streaming é **separada** da Gold oficial | Nunca misturar com `ft_*` oficiais sem filtrar `origem` |

## Batch × Streaming (neste projeto)

| Aspecto | Batch | Streaming simulado |
|---|---|---|
| Fonte | Snapshot completo do BigQuery | Eventos incrementais na fila |
| Frequência | Por execução do Step Functions | Micro-lotes por ciclo do notebook |
| Latência | Duração da carga (`pipeline_runs.duration_seconds`) | `processing_time − event_time` por evento (`streaming_runs`) |
| Garantia | Reexecução idempotente na Silver (dedup por chave) | At-least-once + dedup por `event_id` (efeito *effectively-once* na Silver) |
| Evento ruim | Bloqueia o lote (DQ) | Rejeita o evento e segue |

In [0]:
import os
import random
import sys
import time
import uuid
from datetime import datetime, timezone, timedelta

from pyspark.sql import functions as F

# módulo testável (src/tc_core.py) — o notebook roda a partir da pasta Bronze/Streaming
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
import tc_core  # noqa: E402

dbutils.widgets.text("n_ciclos", "5", "Ciclos produtor→consumidor")
dbutils.widgets.text("eventos_por_ciclo", "20", "Eventos novos por ciclo")
dbutils.widgets.text("espera_segundos", "3", "Espera simulada de transporte (s)")
dbutils.widgets.text("pct_invalidos", "0.08", "Fração de eventos inválidos")
dbutils.widgets.text("pct_reentrega", "0.10", "Fração de reentregas (duplicados)")
dbutils.widgets.dropdown("reset", "false", ["false", "true"], "Recriar tabelas/checkpoint de streaming")

N_CICLOS = int(dbutils.widgets.get("n_ciclos"))
EVENTOS_POR_CICLO = int(dbutils.widgets.get("eventos_por_ciclo"))
ESPERA = float(dbutils.widgets.get("espera_segundos"))
PCT_INVALIDOS = float(dbutils.widgets.get("pct_invalidos"))
PCT_REENTREGA = float(dbutils.widgets.get("pct_reentrega"))
RESET = dbutils.widgets.get("reset") == "true"

FILA = "workspace.bronze.eventos_fila"
RAW = "workspace.bronze.eventos_streaming_raw"
SILVER = "workspace.silver.eventos_streaming"
GOLD = "workspace.gold.ft_indicador_streaming"
RUNS = "workspace.monitoring.streaming_runs"
VOLUME = "workspace.bronze.streaming_checkpoints"
CKPT = "/Volumes/workspace/bronze/streaming_checkpoints/eventos_fila"

## 1. Tabelas (esquema novo; tabelas antigas de streaming com outro esquema são recriadas)
São tabelas **simuladas**; recriá-las não afeta nenhuma tabela oficial.

In [0]:
TABELAS = {
    FILA: """event_id STRING NOT NULL, event_time TIMESTAMP NOT NULL, ano INT, sigla_uf STRING,
             serie INT, rede_codigo STRING, taxa_alfabetizacao DOUBLE, meta_alfabetizacao DOUBLE,
             origem STRING NOT NULL, entrega INT NOT NULL""",
    RAW: """event_id STRING, event_time TIMESTAMP, ano INT, sigla_uf STRING, serie INT,
            rede_codigo STRING, taxa_alfabetizacao DOUBLE, meta_alfabetizacao DOUBLE,
            origem STRING, entrega INT, processing_time TIMESTAMP, latencia_segundos DOUBLE,
            valido BOOLEAN, motivo_invalido STRING, batch_id BIGINT, lote_id STRING""",
    SILVER: """event_id STRING NOT NULL, event_time TIMESTAMP NOT NULL, ano INT, sigla_uf STRING,
               serie INT, rede_codigo STRING, taxa_alfabetizacao DOUBLE, meta_alfabetizacao DOUBLE,
               origem STRING NOT NULL, processing_time TIMESTAMP NOT NULL,
               latencia_segundos DOUBLE NOT NULL, batch_id BIGINT""",
    GOLD: """event_id STRING NOT NULL, event_time TIMESTAMP NOT NULL, ano INT, sigla_uf STRING,
             serie INT, rede_codigo STRING, taxa_alfabetizacao DOUBLE, meta_alfabetizacao DOUBLE,
             diferenca_meta DOUBLE, situacao_meta STRING, origem STRING NOT NULL,
             latencia_segundos DOUBLE NOT NULL, processing_time TIMESTAMP NOT NULL""",
    RUNS: """run_id STRING, micro_batch_id BIGINT, started_at TIMESTAMP, finished_at TIMESTAMP,
             duration_seconds DOUBLE, events_received BIGINT, events_valid BIGINT,
             events_invalid BIGINT, events_duplicated BIGINT, avg_latency_seconds DOUBLE,
             max_latency_seconds DOUBLE, p95_latency_seconds DOUBLE, status STRING,
             error_message STRING""",
}


def colunas_esperadas(ddl):
    return [c.strip().split()[0] for c in ddl.replace("\n", " ").split(",") if c.strip()]


for schema in ("bronze", "silver", "gold", "monitoring"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {VOLUME}")

recriou_fila = False
for nome, ddl in TABELAS.items():
    existe = spark.catalog.tableExists(nome)
    divergente = existe and [c.lower() for c in spark.table(nome).columns] != colunas_esperadas(ddl)
    if RESET or divergente:
        if existe:
            spark.sql(f"DROP TABLE {nome}")
            print(f"recriando {nome} ({'reset' if RESET else 'esquema divergente'})")
        recriou_fila = recriou_fila or nome == FILA
    spark.sql(f"CREATE TABLE IF NOT EXISTS {nome} ({ddl}) USING DELTA")

# checkpoint e fila precisam andar juntos: fila recriada → checkpoint antigo é inválido
if RESET or recriou_fila:
    dbutils.fs.rm(CKPT, True)
    print("checkpoint removido")

## 2. Domínios válidos (vêm das tabelas oficiais, sem lista fixa no código)

In [0]:
UFS_VALIDAS = sorted(r[0] for r in spark.sql(
    "SELECT DISTINCT sigla_uf FROM workspace.silver.uf WHERE sigla_uf IS NOT NULL").collect())
REDES_VALIDAS = sorted(r[0] for r in spark.sql(
    "SELECT DISTINCT CAST(chave AS STRING) FROM workspace.silver.dicionario "
    "WHERE nome_coluna = 'rede' AND chave IS NOT NULL").collect())
if not UFS_VALIDAS or not REDES_VALIDAS:
    raise RuntimeError("Domínios vazios: rode Silver (uf, dicionário) antes do streaming.")
print(f"UFs: {len(UFS_VALIDAS)} | redes: {REDES_VALIDAS}")

## 3. Produtor (grava na fila; `event_time` = instante real de produção)

In [0]:
SCHEMA_FILA = ("event_id string, event_time timestamp, ano int, sigla_uf string, serie int, "
               "rede_codigo string, taxa_alfabetizacao double, meta_alfabetizacao double, "
               "origem string, entrega int")
_historico = []  # eventos já enviados (base para reentrega)


def produzir(n_novos):
    """Envia n_novos eventos novos + reentregas de eventos já enviados (mesmo event_id e event_time)."""
    novos = []
    for _ in range(n_novos):
        invalido = random.random() < PCT_INVALIDOS
        ev = {
            "event_id": str(uuid.uuid4()),
            "event_time": datetime.now(timezone.utc),   # carimbo de PRODUÇÃO (tz-aware)
            "ano": random.choice([2024, 2025]),
            "sigla_uf": random.choice(UFS_VALIDAS),
            "serie": 2,
            "rede_codigo": random.choice(REDES_VALIDAS),
            "taxa_alfabetizacao": round(random.uniform(20, 95), 2),
            "meta_alfabetizacao": round(random.uniform(50, 80), 2),
            "origem": "SIMULADO",
            "entrega": 1,
        }
        if invalido:
            defeito = random.choice(["taxa_nula", "taxa_negativa", "taxa_gt100", "uf", "rede"])
            if defeito == "taxa_nula":
                ev["taxa_alfabetizacao"] = None
            elif defeito == "taxa_negativa":
                ev["taxa_alfabetizacao"] = -1.0
            elif defeito == "taxa_gt100":
                ev["taxa_alfabetizacao"] = 150.0
            elif defeito == "uf":
                ev["sigla_uf"] = "XX"
            else:
                ev["rede_codigo"] = "99"
        novos.append(ev)
        time.sleep(random.uniform(0.0, 0.05))  # eventos não nascem todos no mesmo instante

    reentregas = []
    pool = _historico + novos
    if pool:
        for ev in random.sample(pool, min(len(pool), max(1, int(len(pool) * PCT_REENTREGA)))):
            r = dict(ev)
            r["entrega"] = ev["entrega"] + 1   # mesmo event_id e event_time, nova entrega
            reentregas.append(r)

    _historico.extend(novos)
    lote = novos + reentregas
    (spark.createDataFrame([tuple(e[c.split()[0]] for c in SCHEMA_FILA.split(", ")) for e in lote],
                           SCHEMA_FILA)
        .write.format("delta").mode("append").saveAsTable(FILA))
    return len(novos), len(reentregas)

## 4. Consumidor (`foreachBatch`): valida → Bronze → dedup entre lotes (MERGE) → Silver → Gold → métricas

In [0]:
COLS_EVENTO = ["event_id", "event_time", "ano", "sigla_uf", "serie", "rede_codigo",
               "taxa_alfabetizacao", "meta_alfabetizacao", "origem"]


def _registrar_run(run_id, batch_id, started, status, n_recv=0, n_val=0, n_inv=0, n_dup=0,
                   avg_l=None, max_l=None, p95_l=None, erro=None):
    finished = datetime.now(timezone.utc)
    spark.createDataFrame(
        [(run_id, int(batch_id), started, finished, (finished - started).total_seconds(),
          n_recv, n_val, n_inv, n_dup, avg_l, max_l, p95_l, status, erro)],
        """run_id string, micro_batch_id long, started_at timestamp, finished_at timestamp,
           duration_seconds double, events_received long, events_valid long, events_invalid long,
           events_duplicated long, avg_latency_seconds double, max_latency_seconds double,
           p95_latency_seconds double, status string, error_message string"""
    ).write.format("delta").mode("append").saveAsTable(RUNS)


def processar_lote(batch_df, batch_id):
    sess = batch_df.sparkSession
    run_id, lote_id = str(uuid.uuid4()), str(uuid.uuid4())
    started = datetime.now(timezone.utc)
    try:
        # Carimbo de INGESTÃO: lido do relógio no momento em que o consumidor pega o lote.
        # (Literal Python → mesmo valor em todas as ações do lote; não vem do produtor.)
        chegada = datetime.now(timezone.utc)

        b = (batch_df
             .withColumn("processing_time", F.lit(chegada).cast("timestamp"))
             .withColumn("latencia_segundos",
                         tc_core.latencia_segundos(F.col("event_time"), F.col("processing_time")))
             .withColumn("motivo_invalido",
                         tc_core.motivo_invalido(F.col("taxa_alfabetizacao"), F.col("sigla_uf"),
                                                 F.col("rede_codigo"), F.col("event_id"),
                                                 UFS_VALIDAS, REDES_VALIDAS))
             .withColumn("valido", F.col("motivo_invalido").isNull())
             .withColumn("batch_id", F.lit(int(batch_id)))
             .withColumn("lote_id", F.lit(lote_id)))

        # 1) Bronze: tudo que o broker entregou (inclusive inválidos e reentregas)
        b.select(*COLS_EVENTO, "entrega", "processing_time", "latencia_segundos", "valido",
                 "motivo_invalido", "batch_id", "lote_id").write.format("delta").mode("append").saveAsTable(RAW)

        # 2) Métricas do lote
        m = b.agg(
            F.count("*").alias("n"),
            F.sum(F.when(F.col("valido"), 1).otherwise(0)).alias("n_validos"),
            # latência medida só na 1ª entrega (reentrega mede o atraso da fila, não da produção)
            F.avg(F.when(F.col("entrega") == 1, F.col("latencia_segundos"))).alias("avg_l"),
            F.max(F.when(F.col("entrega") == 1, F.col("latencia_segundos"))).alias("max_l"),
            F.percentile_approx(F.when(F.col("entrega") == 1, F.col("latencia_segundos")), 0.95).alias("p95_l"),
        ).first()
        n_recv, n_val = int(m["n"]), int(m["n_validos"] or 0)

        # 3) Dedup ENTRE lotes: só entra na Silver quem ainda não existe lá
        validos_unicos = b.filter("valido").dropDuplicates(["event_id"])
        novos = (validos_unicos
                 .join(sess.table(SILVER).select("event_id"), "event_id", "left_anti")
                 .select(*COLS_EVENTO, "processing_time", "latencia_segundos", "batch_id"))
        novos.createOrReplaceTempView("_novos_lote")
        n_novos = sess.table("_novos_lote").count()

        sess.sql(f"""
            MERGE INTO {SILVER} t USING _novos_lote s ON t.event_id = s.event_id
            WHEN NOT MATCHED THEN INSERT *
        """)

        # 4) Gold de streaming (apenas os novos; nulo de meta vira SEM_META, nunca zero)
        (sess.table("_novos_lote")
            .withColumn("diferenca_meta", F.col("taxa_alfabetizacao") - F.col("meta_alfabetizacao"))
            .withColumn("situacao_meta", tc_core.situacao_meta(F.col("taxa_alfabetizacao"),
                                                               F.col("meta_alfabetizacao")))
            .select("event_id", "event_time", "ano", "sigla_uf", "serie", "rede_codigo",
                    "taxa_alfabetizacao", "meta_alfabetizacao", "diferenca_meta", "situacao_meta",
                    "origem", "latencia_segundos", "processing_time")
            .write.format("delta").mode("append").saveAsTable(GOLD))

        _registrar_run(run_id, batch_id, started, "SUCESSO", n_recv, n_val, n_recv - n_val,
                       n_val - n_novos, m["avg_l"], m["max_l"], m["p95_l"])
        print(f"lote {batch_id}: recebidos={n_recv} válidos={n_val} inválidos={n_recv - n_val} "
              f"duplicados={n_val - n_novos} novos={n_novos} latência média(1ª entrega)="
              f"{(m['avg_l'] if m['avg_l'] is not None else float('nan')):.2f}s")
    except Exception as exc:
        _registrar_run(run_id, batch_id, started, "FALHA", erro=str(exc)[:2000])
        raise  # propaga: o job/orquestrador precisa ver a falha

## 5. Execução: ciclos produtor → espera simulada → consumidor incremental

In [0]:
random.seed()  # sem semente fixa: latências e defeitos variam de verdade
for ciclo in range(N_CICLOS):
    n_novos, n_re = produzir(EVENTOS_POR_CICLO)
    print(f"ciclo {ciclo}: produzidos {n_novos} novos + {n_re} reentregas")
    time.sleep(ESPERA)  # transporte/atraso de fila simulado (o tempo medido é o relógio real)
    q = (spark.readStream.table(FILA)
         .writeStream.foreachBatch(processar_lote)
         .option("checkpointLocation", CKPT)
         .trigger(availableNow=True)
         .start())
    q.awaitTermination()

## 6. Verificações (falham o notebook se as garantias não valerem)

In [0]:
silver = spark.table(SILVER)
raw = spark.table(RAW)

n_silver = silver.count()
n_unicos = silver.select("event_id").distinct().count()
assert n_silver == n_unicos, f"Silver com event_id duplicado: {n_silver} linhas, {n_unicos} únicos"

assert silver.filter("taxa_alfabetizacao IS NULL OR taxa_alfabetizacao < 0 OR taxa_alfabetizacao > 100").count() == 0, \
    "Evento inválido chegou na Silver"
assert [r[0] for r in silver.select("origem").distinct().collect()] == ["SIMULADO"], "origem diferente de SIMULADO"
assert spark.table(GOLD).filter("origem <> 'SIMULADO'").count() == 0

lat = raw.filter("entrega = 1").agg(F.min("latencia_segundos").alias("mn"), F.avg("latencia_segundos").alias("avg"),
              F.max("latencia_segundos").alias("mx")).first()
assert lat["mn"] > 0, "latência mínima ≤ 0: carimbos de produção e ingestão não são independentes"

dup_entregues = raw.count() - raw.select("event_id").distinct().count()
print(f"Silver: {n_silver} eventos únicos | Bronze recebeu {raw.count()} (reentregas: {dup_entregues})")
print(f"Latência medida: min={lat['mn']:.2f}s média={lat['avg']:.2f}s máx={lat['mx']:.2f}s")
print("Inválidos por motivo:")
raw.filter("NOT valido").groupBy("motivo_invalido").count().show(truncate=False)
spark.table(RUNS).orderBy("started_at").select(
    "micro_batch_id", "events_received", "events_valid", "events_invalid", "events_duplicated",
    "avg_latency_seconds", "max_latency_seconds", "p95_latency_seconds", "status").show(truncate=False)